# Exercise 11 - Cython and Numba

## Cython

In [ ]:
import Cython
%load_ext Cython

# Note: To use Cython on a Windows machine you may need to install Visual Studio to use it 
# to install the Python Extensions Package

#### Create random array
https://docs.python.org/3/library/array.html

In [ ]:
from array import array
from random import randint

In [ ]:
# create random array
arr = array('i', (randint(-1000, 1000) for _ in range(10000000)))

In [ ]:
def sum_py(arr):

    return sum_

In [ ]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_py(arr)

In [ ]:
%%cython


In [ ]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy(arr)

#### Static Typing

In [ ]:
%%cython
def sum_cy_st(arr):

    return sum_

In [ ]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy_st(arr)

#### Typed Memoryviews

In [ ]:
%%cython
def sum_cy_tm_1():

    return sum_

In [ ]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy_st(arr)
%timeit -r3 -n5 sum_cy_tm_1(arr)

#### With C-like looping

In [ ]:
%%cython
from cpython cimport array
def sum_cy_tm_2():

    return sum_

In [ ]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy_st(arr)
%timeit -r3 -n5 sum_cy_tm_2(arr)

#### Compile Optimizations

In [ ]:
%%cython
cimport cython
from cpython cimport array

@cython.boundscheck(False)  # just read from/write to memory without checking if an index is in a given range
@cython.wraparound(False)  # using negative indices is not possible anymore
# see also: https://docs.cython.org/en/latest/src/userguide/source_files_and_compilation.html#compiler-directives
def sum_cy_co():

    return sum_

In [ ]:
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy_co(arr)

In [ ]:
# With Memory view layout --> int[::1] when passing an array as function argument
# see also: https://cython.readthedocs.io/en/latest/src/userguide/memoryviews.html
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_cy_co(arr)

#### Together with Multithreading

In [ ]:
%%cython
cimport cython
from cpython cimport array

@cython.boundscheck(False)
@cython.wraparound(False)
def _sum_gil(int[::1] arr, int[:] out):
    
      
def sum_threads(int[::1] arr):
    
    
    return out[0] + out[1]

In [ ]:
# with GIL
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_threads(arr)

In [ ]:
# with GIL released (use with nogil statement)
%timeit -r3 -n5 sum(arr)
%timeit -r3 -n5 sum_threads(arr)

#### Interfacing with C code
after -I argument we need to specify the path where sum.h is located

In [ ]:
%%cython -I --specify your path here
cimport cython
from cpython cimport array

cdef extern from "sum.h" nogil:
    cdef int sum_c(int* a, int n)
    
@cython.boundscheck(False)
@cython.wraparound(False)
def sum_cy_c(int[::1] a):
    cdef int out
    with nogil: 
        out = sum_c(&a[0], a.shape[0])
    return out

#### Numpy (Vectorization)

In [ ]:
import numpy as np

np_arr = np.asarray(arr)

%timeit -r3 -n5 sum(np_arr)
%timeit -r3 -n5 np_arr.sum()
%timeit -r3 -n5 sum_cy_co(np_arr)

#### Proof that they are all working

In [ ]:
(sum_py(arr), sum_cy(arr), sum(arr), sum_cy_st(arr), sum_cy_tm_1(arr), sum_cy_tm_2(arr), sum_cy_co(arr),
 sum_threads(arr), sum_cy_c(arr), np_arr.sum())

## Numba

In [ ]:
import os
os.environ['NUMBA_NUM_THREADS'] = '10'
import numba as nb
import numpy as np

In [ ]:
arr = np.random.random((4096,4096))

In [ ]:
def py_sum2d(arr):

    return result

@nb.jit
def nb_sum2d(arr):

    return result

In [ ]:
%timeit -r1 -n1 py_sum2d(arr)
%timeit -r1 -n1 nb_sum2d(arr)  # --> first call is slow

In [ ]:
#@nb.jit(nopython=True)
@nb.njit
def nb_sum2d_nopython(arr):

    return result

In [ ]:
%timeit -r5 -n10 nb_sum2d(arr)
%timeit -r5 -n10 nb_sum2d_nopython(arr)

In [ ]:
@nb.jit(nopython=True, parallel=True)
def nb_sum2d_parallel(arr):

    return result

In [ ]:
%timeit -r5 -n10 nb_sum2d(arr)
%timeit -r5 -n10 nb_sum2d_parallel(arr)  # --> add nb.prange (https://numba.pydata.org/numba-doc/0.11/prange.html)

In [ ]:
nb_sum2d_parallel.parallel_diagnostics(level=4)